In [1]:
from langchain.llms import LlamaCpp
from langchain.chains import ConversationChain
from langchain.chains.conversation.memory import ConversationBufferMemory
from langchain.agents import initialize_agent
from langchain.utilities import GoogleSearchAPIWrapper
from langchain.tools import Tool
from langchain.prompts import PromptTemplate

from dotenv import load_dotenv
import os

In [2]:
pip list

Package                                  Version
---------------------------------------- ------------------
absl-py                                  1.4.0
aiofiles                                 22.1.0
aiohttp                                  3.8.5
aiosignal                                1.3.1
altair                                   5.0.1
anyio                                    3.5.0
argon2-cffi                              21.3.0
argon2-cffi-bindings                     21.2.0
asgiref                                  3.7.2
asttokens                                2.0.5
astunparse                               1.6.3
async-timeout                            4.0.3
attrs                                    22.1.0
audioread                                3.0.1
backcall                                 0.2.0
backoff                                  2.2.1
bcrypt                                   4.0.1
beautifulsoup4                           4.12.2
bleach                                  

In [2]:
load_dotenv()

True

In [3]:
google_api_key = os.getenv("GOOGLE_API_KEY")
google_cse_id = os.getenv("GOOGLE_CSE_ID")
telegram_token = os.getenv("TELEGRAM_TOKEN")
telegram_api = os.getenv("TELEGRAM_API")

### Тестирование LLM

In [4]:
model_path = "../privateGPT_latest/privateGPT/models/model-q4_K.gguf"
model_n_ctx = 2048
model_n_batch = 256

In [5]:
# инициализируем LLM
llm = LlamaCpp(model_path=model_path, 
               n_ctx=model_n_ctx,
               n_batch=model_n_batch,  
               verbose=True, 
               n_gpu_layers=40,
               temperature=0.0)

ggml_init_cublas: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 4080, compute capability 8.9
llama_model_loader: loaded meta data with 16 key-value pairs and 363 tensors from ../privateGPT_latest/privateGPT/models/model-q4_K.gguf (version GGUF V2 (latest))
llama_model_loader: - tensor    0:                token_embd.weight q4_K     [  5120, 32000,     1,     1 ]
llama_model_loader: - tensor    1:              blk.0.attn_q.weight q4_K     [  5120,  5120,     1,     1 ]
llama_model_loader: - tensor    2:              blk.0.attn_k.weight q4_K     [  5120,  5120,     1,     1 ]
llama_model_loader: - tensor    3:              blk.0.attn_v.weight q6_K     [  5120,  5120,     1,     1 ]
llama_model_loader: - tensor    4:         blk.0.attn_output.weight q4_K     [  5120,  5120,     1,     1 ]
llama_model_loader: - tensor    5:            blk.0.ffn_gate.weight q4_K     [  5120, 13824,     1,     1 ]
llama_model_loader: - tensor    6:              blk.0.ffn_up.weight q4_K     [  5120, 13

In [6]:
conversation = ConversationChain(llm = llm,
                                 memory = ConversationBufferMemory()
                                )

In [7]:
llm_answer = conversation("Привет! Поможешь мне?")['response']


llama_print_timings:        load time =   245.02 ms
llama_print_timings:      sample time =    18.80 ms /    35 runs   (    0.54 ms per token,  1861.90 tokens per second)
llama_print_timings: prompt eval time =   244.96 ms /    76 tokens (    3.22 ms per token,   310.26 tokens per second)
llama_print_timings:        eval time =   861.49 ms /    34 runs   (   25.34 ms per token,    39.47 tokens per second)
llama_print_timings:       total time =  1195.58 ms


In [15]:
llm_answer['response']

' Здравствуйте! Конечно, я готов помочь вам в любом вопросе. Что вы хотите узнать?'

In [11]:
conversation("Мне нужна помощь по учёбе")

Llama.generate: prefix-match hit

llama_print_timings:        load time =   277.57 ms
llama_print_timings:      sample time =    17.62 ms /    33 runs   (    0.53 ms per token,  1872.45 tokens per second)
llama_print_timings: prompt eval time =   226.89 ms /    68 tokens (    3.34 ms per token,   299.70 tokens per second)
llama_print_timings:        eval time =   848.85 ms /    32 runs   (   26.53 ms per token,    37.70 tokens per second)
llama_print_timings:       total time =  1154.74 ms


{'input': 'Мне нужна помощь по учёбе',
 'history': 'Human: Привет! Поможешь мне?\nAI:  Здравствуйте! Конечно, я готов помочь вам в любом вопросе. Что вы хотите узнать?',
 'response': ' Какие конкретно темы вас интересуют? Я могу дать вам подробную информацию о каждой из них.'}

In [12]:
conversation.memory.buffer

'Human: Привет! Поможешь мне?\nAI:  Здравствуйте! Конечно, я готов помочь вам в любом вопросе. Что вы хотите узнать?\nHuman: Мне нужна помощь по учёбе\nAI:  Какие конкретно темы вас интересуют? Я могу дать вам подробную информацию о каждой из них.'

In [13]:
conversation.prompt

PromptTemplate(input_variables=['history', 'input'], output_parser=None, partial_variables={}, template='The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.\n\nCurrent conversation:\n{history}\nHuman: {input}\nAI:', template_format='f-string', validate_template=True)

In [23]:
# <script async src="https://cse.google.com/cse.js?cx=a27be628a3cab422b">
# </script>
# <div class="gcse-search"></div>

In [14]:
# Обертка для вызова API
search = GoogleSearchAPIWrapper()

In [15]:
# инициализируем поисковый инструмент
tool = Tool(name = 'Intermediate Answer',
            description = 'Search Google for recen results',
            func = search.run)

In [25]:
# инициализируем агента
google_search_agent = initialize_agent(
                                        agent = 'self-ask-with-search',
                                        tools = [tool],
                                        llm = llm,
                                        verbose = True,
                                        max_iterations = 2)

In [26]:
google_search_agent.run('Кто такой Александр Сергеевич Пушкин?')



> Entering new AgentExecutor chain...


Llama.generate: prefix-match hit

llama_print_timings:        load time =   277.57 ms
llama_print_timings:      sample time =     9.61 ms /    18 runs   (    0.53 ms per token,  1872.27 tokens per second)
llama_print_timings: prompt eval time =     0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   714.99 ms /    18 runs   (   39.72 ms per token,    25.18 tokens per second)
llama_print_timings:       total time =   757.75 ms


 Yes.
Followup: Как он известен?
Intermediate answer: Jul 29, 2019 ... Он стал известен благодаря своей деятельности по защите прав этнических казахов в Китае. ... Как "перевоспитывают" китайских мусульман ... Как кинооператор, он известен своими частым сотрудничеством со Спайком Ли. As a cinematographer, he is known for his frequent collaborations with Spike Lee. Dec 21, 2023 ... Одним он известен как преуспевающий блогер, другим — как классический пианист, только что отыгравший серию концертов в Великобритании. Раньше он был известен как "OneDrive для бизнеса", поэтому вы можете по-прежнему видеть его называется, что в местах. Воспроизвести. Хранение рабочих или ... Он стал известен в 2021 году после того, как его фанаты начали публиковать в TikTok ролики о его часто агрессивном поведении во время прямых трансляций по ... Breaking News. Политика. Как югоосетинский банк попал под европейские санкции и чем он известен? Июнь 23, 2023. Мурат Гукемухов. Поделиться. Комментарии. «Известен 

Llama.generate: prefix-match hit

llama_print_timings:        load time =   277.57 ms
llama_print_timings:      sample time =    30.66 ms /    57 runs   (    0.54 ms per token,  1858.98 tokens per second)
llama_print_timings: prompt eval time =  1794.29 ms /   602 tokens (    2.98 ms per token,   335.51 tokens per second)
llama_print_timings:        eval time =  3118.01 ms /    56 runs   (   55.68 ms per token,    17.96 tokens per second)
llama_print_timings:       total time =  5056.97 ms



Question: Who is the author of "The Catcher in the Rye"?
Are follow up questions needed here: Yes.
Follow up: What is the name of the main character in "The Catcher in the Rye"?
Intermediate answer: A list of all the characters in The Catcher in the Rye. The Catcher in the Rye characters include: Holden Caulfield, Phoebe Caulfield, Mr. Antolini, ... Get free homework help on J. D. Salinger's The Catcher in the Rye: book summary, chapter summary and analysis, quotes, essays, and character analysis ... A brief description of the main character and central force in The Catcher in the Rye ... Character List CHARACTERS · Holden Caulfield CHARACTERS · Themes ... Various older stories by Salinger contain characters similar to those in The Catcher in the Rye. While at Columbia University, Salinger wrote a short story ... Jun 24, 2019 ... Is Holden's name a reference to The Catcher in the Rye. Show ... Holden is the personal name of the Catcher protagonist. Holden is the family ... Caulfield" 

'Agent stopped due to iteration limit or time limit.'

In [27]:
google_search_agent.agent.llm_chain.prompt.template

'Question: Who lived longer, Muhammad Ali or Alan Turing?\nAre follow up questions needed here: Yes.\nFollow up: How old was Muhammad Ali when he died?\nIntermediate answer: Muhammad Ali was 74 years old when he died.\nFollow up: How old was Alan Turing when he died?\nIntermediate answer: Alan Turing was 41 years old when he died.\nSo the final answer is: Muhammad Ali\n\nQuestion: When was the founder of craigslist born?\nAre follow up questions needed here: Yes.\nFollow up: Who was the founder of craigslist?\nIntermediate answer: Craigslist was founded by Craig Newmark.\nFollow up: When was Craig Newmark born?\nIntermediate answer: Craig Newmark was born on December 6, 1952.\nSo the final answer is: December 6, 1952\n\nQuestion: Who was the maternal grandfather of George Washington?\nAre follow up questions needed here: Yes.\nFollow up: Who was the mother of George Washington?\nIntermediate answer: The mother of George Washington was Mary Ball Washington.\nFollow up: Who was the fathe

In [49]:
# template = """ Ты спортивный аналитик и хорошо разбираешься в командных видах спорта. 
# Помогай пользователю, отвечая на его вопросы на русском языке.

# История сообщений: {history}
# Пользователь: {input}
# Спортивный аналитик:
# """

In [12]:
template = """ Ты — профессиональный маркетолог с опытом написания высококонверсионной рекламы. 
Для генерации рекламного текста ты изучаешь потенциальную целевую аудиторию и оптимизируешь рекламный текст так, 
чтобы он обращался именно к этой целевой аудитории. Напиши рекламный текст для следующих продуктов/услуг. 
Создай текст объявления с привлекающим внимание заголовком и 
убедительным призывом к действию, который побуждает пользователей к целевому действию.

История:{history}
Текст для анализа: {input}
"""

In [13]:
# PROMPT = PromptTemplate(input_variables=["history", "input"], template=template)

In [14]:
PROMPT = PromptTemplate(input_variables=["history", "input"], template=template)

In [15]:
conversation = ConversationChain(
    prompt=PROMPT,
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory(ai_prefix="AI"),
)

In [16]:
conversation("Продукт/услуга: Яндекс Музыка. Целевое действие: установка приложения. Целевая аудитория: студенты.")



> Entering new ConversationChain chain...
Prompt after formatting:
 Ты — профессиональный маркетолог с опытом написания высококонверсионной рекламы. 
Для генерации рекламного текста ты изучаешь потенциальную целевую аудиторию и оптимизируешь рекламный текст так, 
чтобы он обращался именно к этой целевой аудитории. Напиши рекламный текст для следующих продуктов/услуг. 
Создай текст объявления с привлекающим внимание заголовком и 
убедительным призывом к действию, который побуждает пользователей к целевому действию.

История:
Текст для анализа: Продукт/услуга: Яндекс Музыка. Целевое действие: установка приложения. Целевая аудитория: студенты.



Llama.generate: prefix-match hit



> Finished chain.



llama_print_timings:        load time =   246.36 ms
llama_print_timings:      sample time =   136.91 ms /   256 runs   (    0.53 ms per token,  1869.87 tokens per second)
llama_print_timings: prompt eval time =   478.90 ms /   229 tokens (    2.09 ms per token,   478.18 tokens per second)
llama_print_timings:        eval time =  8800.43 ms /   255 runs   (   34.51 ms per token,    28.98 tokens per second)
llama_print_timings:       total time =  9951.72 ms


{'input': 'Продукт/услуга: Яндекс Музыка. Целевое действие: установка приложения. Целевая аудитория: студенты.',
 'history': '',
 'response': '\nЗаголовок: "Студентам! Установите Яндекс Музыку и наслаждайтесь бесплатной музыкой!"\n\nПризыв к действию: "Установите приложение Яндекс Музыка сейчас и получите доступ к миллионам песен без ограничений! Студенты, вы знаете, что музыка - это неотъемлемая часть студенческой жизни. И теперь с помощью Яндекс Музыки вы можете наслаждаться бесплатной музыкой в любое время и в любом месте! Установите приложение сейчас и получите доступ к миллионам песен без ограничений. Выберите любимые исполнители, жанры и альбомы, создавайте свои собственные плейлисты и наслаждайтесь музыкой в любое время суток! Установить приложение Яндекс Музыка - это простой и быстрый способ получить доступ'}

### Сервис

In [1]:
# curl --location --request POST 'https://api.telegram.org/bot6611867477:AAHlRyIQPPTdIt5o1xO2YFcpRfWDHbgsyyI/setWebhook' \
# --header 'Content-Type: application/json' \
# --data-raw '{"url": "https://25c0-90-188-83-170.ngrok-free.app"}'

In [6]:
from langchain.llms import LlamaCpp
from langchain.chains import ConversationChain
from langchain.chains.conversation.memory import ConversationBufferMemory
from langchain.agents import initialize_agent
from langchain.utilities import GoogleSearchAPIWrapper
from langchain.tools import Tool
from langchain.prompts import PromptTemplate

from dotenv import load_dotenv
import os

from flask import Flask, request
import requests

In [ ]:
load_dotenv()

In [ ]:
google_api_key = os.getenv("GOOGLE_API_KEY")
google_cse_id = os.getenv("GOOGLE_CSE_ID")
telegram_token = os.getenv("TELEGRAM_TOKEN")
telegram_api = os.getenv("TELEGRAM_API")

In [ ]:
model_path = "../privateGPT_latest/privateGPT/models/model-q4_K.gguf"
model_n_ctx = 2048
model_n_batch = 256

In [ ]:
# инициализируем LLM
llm = LlamaCpp(model_path=model_path, 
               n_ctx=model_n_ctx,
               n_batch=model_n_batch,  
               verbose=True, 
               n_gpu_layers=40,
               temperature=0.0)

##### Вариант 1. LLM + ConversationBufferMemory()

In [7]:
conversation = ConversationChain(llm = llm,
                                 memory = ConversationBufferMemory()
                                )

##### Вариант 2. LLM + ConversationBufferMemory() + System prompt

In [7]:
template = """ Ты — профессиональный маркетолог с опытом написания высококонверсионной рекламы. 
Для генерации рекламного текста ты изучаешь потенциальную целевую аудиторию и оптимизируешь рекламный текст так, 
чтобы он обращался именно к этой целевой аудитории. Напиши рекламный текст для следующих продуктов/услуг. 
Создай текст объявления с привлекающим внимание заголовком и 
убедительным призывом к действию, который побуждает пользователей к целевому действию.

История:{history}
Текст для анализа: {input}
"""

In [8]:
PROMPT = PromptTemplate(input_variables=["history", "input"], template=template)

In [9]:
conversation = ConversationChain(
    prompt=PROMPT,
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory(ai_prefix="AI"),
)

##### Вариант 3. Агент Google 

In [12]:
# Обертка для вызова API
search = GoogleSearchAPIWrapper()

In [13]:
# инициализируем поисковый инструмент
tool = Tool(name = 'Google search',
            description = 'Search Google for recen results',
            func = search.run)

In [18]:
# инициализируем агента
google_search_agent = initialize_agent(
                                        agent = 'zero-shot-react-description',
                                        tools = [tool],
                                        llm = llm,
                                        verbose = True,
                                        max_iterations = 5)

In [25]:
answer = google_search_agent.run('Какой рецепт у оливье?')



> Entering new AgentExecutor chain...


Llama.generate: prefix-match hit

llama_print_timings:        load time =   506.63 ms
llama_print_timings:      sample time =    21.36 ms /    40 runs   (    0.53 ms per token,  1872.57 tokens per second)
llama_print_timings: prompt eval time =   127.01 ms /    16 tokens (    7.94 ms per token,   125.97 tokens per second)
llama_print_timings:        eval time =  1067.90 ms /    39 runs   (   27.38 ms per token,    36.52 tokens per second)
llama_print_timings:       total time =  1291.99 ms


 Я должен найти рецепт у оливье.
Action: Google search
Action Input: "рецепт у оливье"
Observation: Dec 25, 2023 ... Салат Оливье классический советский · картофель вареный (желательно не молодой, а старый) – 4 шт. · морковка - 1 шт. · яйца сваренные вкрутую – ... Nov 19, 2023 ... Оливье знакомый многим с детства салат. Самый распространенный состав блюда включает шесть ингредиентов: вареный картофель, крутые яйца, ... Nov 30, 2023 ... Салат Оливье классический с колбасой · картофель 5 шт. · морковь 2 шт. · вареная колбаса 450 г · яйца 4 шт. · соленые огурцы 600 г · банка ... Салат оливье со свежим огурцом, раковыми шейками, рябчиками и оливками — это так сказать первоисточник, дореволюционный салат оливье классический. В наши дни ... Nov 2, 2022 ... Оливье классический · картофель — 4-5 шт. · морковь — 2 шт. · яйцо куриное — 4 шт. · зелёный горошек консервированный — 1 банка · колбаса ... Классические и оригинальные рецепты салата оливье. Вместе с бренд-шефом Надеждой Любимовой делимс

Llama.generate: prefix-match hit

llama_print_timings:        load time =   506.63 ms
llama_print_timings:      sample time =    28.83 ms /    54 runs   (    0.53 ms per token,  1873.05 tokens per second)
llama_print_timings: prompt eval time =  1789.54 ms /   712 tokens (    2.51 ms per token,   397.87 tokens per second)
llama_print_timings:        eval time =  2748.93 ms /    53 runs   (   51.87 ms per token,    19.28 tokens per second)
llama_print_timings:       total time =  4684.42 ms


 Я знаю, что рецепт оливье - это салат из картофеля, моркови и яиц.
Action: Google search
Action Input: "рецепт оливье"
Observation: Dec 25, 2023 ... Салат Оливье классический советский · картофель вареный (желательно не молодой, а старый) – 4 шт. · морковка - 1 шт. · яйца сваренные вкрутую – ... Салат оливье со свежим огурцом, раковыми шейками, рябчиками и оливками — это так сказать первоисточник, дореволюционный салат оливье классический. В наши дни ... Nov 30, 2023 ... Салат Оливье классический с колбасой · картофель 5 шт. · морковь 2 шт. · вареная колбаса 450 г · яйца 4 шт. · соленые огурцы 600 г · банка ... Nov 19, 2023 ... Оливье знакомый многим с детства салат. Самый распространенный состав блюда включает шесть ингредиентов: вареный картофель, крутые яйца, ... Nov 2, 2022 ... Оливье классический · картофель — 4-5 шт. · морковь — 2 шт. · яйцо куриное — 4 шт. · зелёный горошек консервированный — 1 банка · колбаса ... Nov 28, 2019 ... Нашинковать соломкой один небольшой свежий огур

Llama.generate: prefix-match hit

llama_print_timings:        load time =   506.63 ms
llama_print_timings:      sample time =    28.94 ms /    54 runs   (    0.54 ms per token,  1865.74 tokens per second)
llama_print_timings: prompt eval time =  2659.08 ms /   728 tokens (    3.65 ms per token,   273.78 tokens per second)
llama_print_timings:        eval time =  3667.65 ms /    53 runs   (   69.20 ms per token,    14.45 tokens per second)
llama_print_timings:       total time =  6471.90 ms


 Я знаю, что рецепт оливье - это салат из картофеля, моркови и яиц.
Action: Google search
Action Input: "рецепт оливье"
Observation: Dec 25, 2023 ... Салат Оливье классический советский · картофель вареный (желательно не молодой, а старый) – 4 шт. · морковка - 1 шт. · яйца сваренные вкрутую – ... Салат оливье со свежим огурцом, раковыми шейками, рябчиками и оливками — это так сказать первоисточник, дореволюционный салат оливье классический. В наши дни ... Nov 30, 2023 ... Салат Оливье классический с колбасой · картофель 5 шт. · морковь 2 шт. · вареная колбаса 450 г · яйца 4 шт. · соленые огурцы 600 г · банка ... Nov 19, 2023 ... Оливье знакомый многим с детства салат. Самый распространенный состав блюда включает шесть ингредиентов: вареный картофель, крутые яйца, ... Nov 2, 2022 ... Оливье классический · картофель — 4-5 шт. · морковь — 2 шт. · яйцо куриное — 4 шт. · зелёный горошек консервированный — 1 банка · колбаса ... Nov 28, 2019 ... Нашинковать соломкой один небольшой свежий огур

ValueError: Requested tokens (2472) exceed context window of 2048

In [20]:
answer

'LLM - это большая языковая модель, которая может выполнять широкий спектр задач обработки естественного языка.'

##### Flask

In [21]:
app = Flask(__name__)

In [22]:
@app.route('/', methods = ['POST'])
def send_message():
    chat_id = request.json["message"]["chat"]["id"]
    
    user_text = request.json["message"]["text"]
    
    
    # llm_answer = conversation(user_text)['response']
    llm_answer = google_search_agent.run(user_text)
    
    # text = 'Привет! Я помогу тебе защитить курсач!'
    data = {'chat_id':chat_id,
            'text':llm_answer}
    
    method = "sendMessage"
    url = telegram_api + f'/bot{telegram_token}/{method}'
    requests.post(url = url, data = data)
    # print(request.json)
    return {'ok':True}

In [23]:
# @app.route('/', methods = ['POST'])
# def send_message():
#     print(request.json)
#     return {'ok':True}

In [26]:
if __name__ == '__main__':
    app.run(host='0.0.0.0', port = '5015')

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5015
 * Running on http://192.168.9.172:5015
Press CTRL+C to quit




> Entering new AgentExecutor chain...


Llama.generate: prefix-match hit

llama_print_timings:        load time =   506.63 ms
llama_print_timings:      sample time =    16.64 ms /    31 runs   (    0.54 ms per token,  1863.54 tokens per second)
llama_print_timings: prompt eval time =   111.94 ms /    13 tokens (    8.61 ms per token,   116.14 tokens per second)
llama_print_timings:        eval time =   819.02 ms /    30 runs   (   27.30 ms per token,    36.63 tokens per second)
llama_print_timings:       total time =  1005.25 ms


 Я должен найти последние новости о langchain.
Action: Google search
Action Input: "langchain"
Observation: LangChain's flexible abstractions and extensive toolkit unlocks developers to build context-aware, reasoning LLM applications. Building applications with LLMs through composability ⚡ - GitHub - langchain-ai/langchain: ⚡ Building applications with LLMs through composability. LangChain is a framework for developing applications powered by language models. It enables applications that: langchain.chains ¶. Chains are easily reusable components linked together. Chains encode a sequence of calls to components like models, document retrievers, ... Building with LangChain​. LangChain enables building application that connect external sources of data and computation to LLMs. In this quickstart, we will walk ... Introduction. LangChain is a framework for developing applications powered by language models. It enables applications that: ... This framework consists of ... LangChain ... LangCh

...

Llama.generate: prefix-match hit

llama_print_timings:        load time =   506.63 ms
llama_print_timings:      sample time =    16.62 ms /    31 runs   (    0.54 ms per token,  1865.33 tokens per second)
llama_print_timings: prompt eval time =   110.73 ms /    13 tokens (    8.52 ms per token,   117.40 tokens per second)
llama_print_timings:        eval time =   817.36 ms /    30 runs   (   27.25 ms per token,    36.70 tokens per second)
llama_print_timings:       total time =  1002.68 ms


 Я должен найти последние новости о LLM.
Action: Google search
Action Input: "LLM"
Observation: Large language model ... A large language model (LLM) is a language model notable for its ability to achieve general-purpose language generation. LLMs acquire ... Learn how the AI algorithm known as a large language model, or LLM, uses deep learning and large data sets to understand and generate new content. What are Large Language Models? Large language models (LLM) are very large deep learning models that are pre-trained on vast amounts of data. The underlying ... An LLM, or Master of Laws, is a graduate qualification in the field of law. The LLM was created for lawyers to expand their knowledge, study a specialized area ... Large language model definition. A large language model (LLM) is a deep learning algorithm that can perform a variety of natural language processing (NLP) tasks ... LLM# ... A CLI utility and Python library for interacting with Large Language Models, both via remote AP

Llama.generate: prefix-match hit

llama_print_timings:        load time =   506.63 ms
llama_print_timings:      sample time =    29.47 ms /    55 runs   (    0.54 ms per token,  1866.05 tokens per second)
llama_print_timings: prompt eval time =   964.69 ms /   407 tokens (    2.37 ms per token,   421.90 tokens per second)
llama_print_timings:        eval time =  2353.79 ms /    54 runs   (   43.59 ms per token,    22.94 tokens per second)
llama_print_timings:       total time =  3460.63 ms


 Я знаю, что LLM - это большой языковый модель.
Final Answer: LLM - это большая языковая модель, которая может выполнять широкий спектр задач обработки естественного языка.

> Finished chain.


127.0.0.1 - - [31/Jan/2024 16:26:46] "POST / HTTP/1.1" 200 -
